<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); border-radius: 16px; padding: 40px; margin-bottom: 24px;">
  <div style="display: flex; align-items: center; gap: 24px;">
    <img src="../docs/assets/logo.png" alt="Gradients" style="height: 80px;">
    <div>
      <h1 style="color: #fff; margin: 0; font-size: 2em;">Gradients Quick-Start</h1>
      <p style="color: #a8b2d1; margin: 8px 0 0 0; font-size: 1.1em;">Instruct, DPO, and Image training — pick a task type and go.</p>
    </div>
  </div>
</div>

## Setup

In [ ]:
%pip install -q --upgrade gradientsio

import os
from gradientsio import GradientsClient, TaskType

os.environ["GRADIENTS_API_KEY"] = os.getenv("GRADIENTS_API_KEY") or input("Paste your Gradients API key: ").strip()
client = GradientsClient()

The helper below renders a styled status card for any training task.

In [ ]:
from IPython.display import HTML, display

def show_status(task):
    d = task.refresh()
    color = {"completed": "#22c55e", "failed": "#ef4444"}.get(str(d.status).lower(), "#eab308")
    model_link = f'<a href="https://huggingface.co/{d.trained_model_repository}" style="color:#7c3aed">{d.trained_model_repository}</a>' if d.trained_model_repository else "<em>training…</em>"
    display(HTML(f"""
        <div style="border:1px solid #e2e8f0; border-radius:12px; padding:20px; margin:12px 0; font-family:system-ui,sans-serif;">
          <div style="display:flex; align-items:center; gap:8px; margin-bottom:8px;">
            <span style="display:inline-block; width:10px; height:10px; border-radius:50%; background:{color};"></span>
            <strong>{d.status}</strong>
          </div>
          <div style="font-size:0.85em; color:#6b7280;">Task ID: <code>{task.task_id}</code></div>
          <div style="margin-top:8px;">Model: {model_link}</div>
        </div>
    """))

## Instruct Task

Fine-tune on instruction/input/output triplets.

In [ ]:
task = client.train(
    model="Qwen/Qwen2.5-7B-Instruct",
    task_type=TaskType.INSTRUCT,
    hours=1,
    dataset="yahma/alpaca-cleaned",
    field_instruction="instruction",
    field_input="input",
    field_output="output",
)
show_status(task)

## DPO Task

Train from preference pairs — prompt, chosen response, rejected response.

In [ ]:
task = client.train(
    model="Qwen/Qwen2.5-7B-Instruct",
    task_type=TaskType.DPO,
    hours=1,
    dataset="trillionlabs/NemoSlides-DPO-mix-v1.0",
    field_prompt="prompt",
    field_chosen="chosen",
    field_rejected="rejected",
)
show_status(task)

## Image LoRA Task

Provide a public/presigned zip URL containing 10-50 image/caption pairs (`.png` + `.txt` with matching names).

In [ ]:
IMAGE_DATASET_ZIP_URL = "https://example.com/my-image-lora-dataset.zip"  # <-- replace this

task = client.tasks.create_image_zip(
    model_repo="stabilityai/stable-diffusion-xl-base-1.0",
    hours_to_complete=1,
    model_type="sdxl",
    ds=IMAGE_DATASET_ZIP_URL,
)
show_status(task)

## Check Any Task

Paste a task ID from a previous run to check its status.

In [ ]:
task = client.tasks.handle("paste-task-id-here")
show_status(task)